In [ ]:
# Final missing-data checks
#
# - Audit the combined real quotes before interpolation.
# - Audit the analysis series again after interpolation.
# - Gaps <= 300 seconds are eligible for interpolation.
# - Gaps > 300 seconds remain unfilled dead zones.
# - Keep the first-two-hour and last-hour exclusions.
# - Keep weekend closures and January 1 excluded.
#
# Required results
# - Missing required 10-second interpolation points: 0.
# - Eligible short gaps left unresolved: 0.
# - Interpolated points inside dead zones: 0.
#
# Dead-zone summary
# - Show each zone's start, end, and actual duration.
# - Show counts and excluded time by year and month.
#
# Measure eligibility between real quotes before applying
# daily exclusions. Exclusions must not turn a long gap
# into a short gap eligible for interpolation.

In [ ]:
# Notebook: 01_etl_and_clean_quotes.ipynb
#
# Objective
# - Combine original CSVs and downloaded replacement exports.
# - Preserve the original files.
# - Build one observed quote history.
# - Fill eligible gaps for later analysis.
# - Report dead zones and check the saved outputs.
#
# Prerequisites
# - Open this notebook from the repository's notebooks folder.
# - Use the current Python environment.
# - Notebook 00 and its saved reports must be available.
#
# Inputs
# - Original EURUSD CSVs directly inside data/.
# - New exports inside data/2017 2018 investigation/.
# - Notebook 00's inventory, bucket definitions and closure records.
#
# Outputs
# - Monthly observed Parquet files.
# - Monthly analysis Parquet files.
# - Source records, reconciliation tables and dead-zone reports.
# - Updated project documentation.
#
# Units
# - Prices: USD per EUR.
# - Sizes: unchanged source units.
# - Displayed sizes are not executed trade volumes.
# - Absolute size denomination remains unconfirmed.
# - Integer timestamps: UTC Unix nanoseconds.
# - Human-readable timestamps: Europe/Athens.
#
# Important
# - Interpolation is for features and regime research only.
# - Synthetic prices and sizes cannot support simulated executions.
# - A synthetic point becomes available only when right anchor B arrives.
# - Future feature code must respect that availability time.

In [ ]:
# Settings and environment.
# Start Jupyter from the repository or its notebooks folder.

from pathlib import Path
import hashlib
import importlib.util
import json
import platform
import shutil
import sys
import tempfile
import time

import duckdb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display

STARTED = time.perf_counter()
FINAL_AUDIT_PASSED = False

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

assert (REPO / "data").is_dir(), "Start Jupyter inside this repository."
assert (REPO / "notebooks").is_dir(), "Repository notebooks folder not found."

RUN_ID = "etl_01_v1"
AUDIT_RUN = "20261029T002746_883835Z"

SOURCE_TZ = "Europe/Athens"
RESEARCH_TZ = "Europe/Athens"

MAX_GAP_SECONDS = 300
INTERPOLATION_STEP_SECONDS = 10

CSV_BLOCK_MIB = 16
DUCKDB_MEMORY_LIMIT = "2GB"
DUCKDB_THREADS = 2

# This is a disk preflight setting, not a storage guarantee.
MIN_FREE_GIB = 40

DATA = REPO / "data"
NEW_EXPORTS = DATA / "2017 2018 investigation"
AUDIT = REPO / "reports" / "tables" / "data_audit" / AUDIT_RUN

RUN = DATA / "processed" / RUN_ID
REPORT = REPO / "reports" / "tables" / "etl" / RUN_ID
FIGURES = REPO / "reports" / "figures" / "etl" / RUN_ID
HELPER = REPO / "functions" / "etl" / "clean_quotes.py"

VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pa.__version__,
    "duckdb": duckdb.__version__,
    "matplotlib": matplotlib.__version__,
}

print("Repository:", REPO)
print("Kernel:", sys.executable)
display(pd.Series(VERSIONS, name="version"))

assert MAX_GAP_SECONDS == 300
assert INTERPOLATION_STEP_SECONDS == 10
assert AUDIT.is_dir(), f"Missing audit reports: {AUDIT}"
assert NEW_EXPORTS.is_dir(), f"Missing export folder: {NEW_EXPORTS}"

In [ ]:
# Agreed processing rules.
#
# Observed quotes
# - Equal timestamp and equal four quote fields: keep one observed copy.
# - Keep the source records so every copy remains traceable.
# - Same prices at different timestamps: retain those observations.
# - Conflicting quote fields at one timestamp: save evidence and stop.
# - Invalid timestamps, prices or sizes: stop for review.
# - Retain locked quotes and zero sizes with flags.
# - Do not remove observations merely because they are extreme.
#
# Gap eligibility
# - Measure each gap between consecutive real quotes.
# - Use actual elapsed time before applying exclusions.
# - Gap <= 300 seconds: eligible.
# - Gap > 300 seconds: never interpolate.
# - Removing excluded hours cannot shorten a gap into eligibility.
#
# Interpolation
# - Keep right anchor B fixed.
# - Insert A+10 seconds, A+20 seconds and so on, strictly before B.
# - Interpolate bid, ask and both displayed sizes linearly.
# - Exactly 300 seconds produces 29 interior points.
# - Every synthetic point has available_at = B.
#
# Analysis exclusions
# - Weekend: Friday 17:00 to Sunday 17:00 New York.
# - First two hours: 17:00–19:00 New York, Sunday–Thursday.
# - Last hour: 16:00–17:00 New York, Monday–Friday.
# - January 1: full Athens calendar day.
# - Apply any separately verified quote-feed closures.
# - Other settlement holidays do not automatically remove trading hours.
#
# Provisional boundary assumption
# - Real anchors may fall just inside excluded hours.
# - Only generated points outside exclusions enter the analysis output.
# - This assumption remains visible for human review.
#
# Continuity
# - Dead zones and exclusions break analysis continuity.
# - Keep both data_segment_id and scope_id.
# - Later rolling calculations must respect both identifiers.

In [ ]:
# Shared helpers: files, timestamps and small checks.
# The following cells collect the helper source.
# Cell 10 writes it to functions/etl/clean_quotes.py.

HELPER_PARTS = [r'''
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

TZ = "Europe/Athens"
NY = "America/New_York"
STEP_NS = 10_000_000_000
LIMIT_NS = 300_000_000_000
BUCKETS = []

VALUES = ["bid", "ask", "bid_size", "ask_size"]

def dump_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False),
        encoding="utf-8"
    )
    temporary.replace(path)

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def write_query(con, query, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial.parquet")
    temporary.unlink(missing_ok=True)
    con.execute(
        f"COPY ({query}) TO {literal(temporary)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD)"
    )
    temporary.replace(path)

def record(con, query):
    cursor = con.execute(query)
    row = cursor.fetchone()
    if row is None:
        return None
    return dict(zip([item[0] for item in cursor.description], row))

def verify_parquet(path, rows):
    assert Path(path).is_file(), f"Missing output: {path}"
    actual = pq.ParquetFile(path).metadata.num_rows
    assert actual == rows, f"Row count changed: {path}"

def athens(value):
    if value is None:
        return None
    return pd.Timestamp(
        int(value), unit="ns", tz="UTC"
    ).tz_convert(TZ).isoformat()

def month_bounds(label):
    start = pd.Timestamp(f"{label}-01", tz=TZ)
    end = start + pd.offsets.MonthBegin(1)
    return int(start.value), int(end.value)

def histogram(con, relation):
    counts = [0] * len(BUCKETS)
    clauses = " ".join(
        f"WHEN gap_ns <= {upper} THEN {index}"
        for index, upper in BUCKETS
    )
    rows = con.execute(
        f"SELECT CASE {clauses} END AS bucket_index, count(*) "
        f"FROM {relation} WHERE gap_ns > 0 GROUP BY 1"
    ).fetchall()
    for index, count in rows:
        assert index is not None
        counts[index] = count
    return counts
''']

In [ ]:
# Shared helpers: exclusion windows and dead-zone pieces.

HELPER_PARTS.append(r'''
def merge_windows(rows):
    merged = []
    for start, end in sorted(rows):
        start, end = int(start), int(end)
        assert end > start
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

def allowed_and_scope(times, windows):
    times = np.asarray(times, dtype=np.int64)
    if not windows:
        return np.ones(len(times), dtype=bool), np.zeros(len(times), dtype=np.int64)
    starts = np.asarray([x[0] for x in windows], dtype=np.int64)
    ends = np.asarray([x[1] for x in windows], dtype=np.int64)
    previous = np.searchsorted(starts, times, side="right") - 1
    inside = (previous >= 0) & (times < ends[np.maximum(previous, 0)])
    scope = np.searchsorted(ends, times, side="right").astype(np.int64)
    return ~inside, scope

def make_windows(first_ns, last_ns, verified_path):
    first = pd.Timestamp(first_ns, unit="ns", tz="UTC").tz_convert(NY).date()
    last = pd.Timestamp(last_ns, unit="ns", tz="UTC").tz_convert(NY).date()
    day, stop = first - timedelta(days=10), last + timedelta(days=10)
    details = []

    def add(start, end, kind):
        details.append({
            "start_ns": int(start.value),
            "end_ns": int(end.value),
            "kind": kind
        })

    while day <= stop:
        if day.weekday() == 4:
            start = pd.Timestamp(f"{day} 17:00", tz=NY)
            end = pd.Timestamp(f"{day + timedelta(days=2)} 17:00", tz=NY)
            add(start, end, "expected_weekend")
        if day.weekday() in (0, 1, 2, 3, 4, 6):
            add(
                pd.Timestamp(f"{day} 16:00", tz=NY),
                pd.Timestamp(f"{day} 19:00", tz=NY),
                "research_daily_exclusion"
            )
        day += timedelta(days=1)

    for year in range(first.year - 1, last.year + 2):
        add(
            pd.Timestamp(f"{year}-01-01", tz=TZ),
            pd.Timestamp(f"{year}-01-02", tz=TZ),
            "january_1_exclusion"
        )

    verified = pd.read_csv(verified_path)
    for row in verified.to_dict("records"):
        start, end = pd.Timestamp(row["start"]), pd.Timestamp(row["end"])
        assert start.tzinfo is not None and end.tzinfo is not None
        add(start, end, "verified_quote_closure")

    windows = merge_windows(
        (row["start_ns"], row["end_ns"]) for row in details
    )
    return windows, details

def allowed_pieces(start, end, windows):
    current = int(start)
    for left, right in windows:
        if right <= current:
            continue
        if left >= end:
            break
        if left > current:
            yield current, min(left, end)
        current = max(current, right)
        if current >= end:
            return
    if current < end:
        yield current, int(end)

def split_by_month(start, end):
    current = int(start)
    while current < end:
        local = pd.Timestamp(current, unit="ns", tz="UTC").tz_convert(TZ)
        label = f"{local.year:04d}-{local.month:02d}"
        _, boundary = month_bounds(label)
        piece_end = min(int(end), boundary)
        yield label, current, piece_end
        current = piece_end
''')

# Shared helpers: stream every CSV into source/month Parquet files.
# Source records are retained for provenance.
# An interrupted source is rebuilt; completed sources can be reused.

HELPER_PARTS.append(r'''
def stage_source(repo, source, directory, block_bytes):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    done_path = directory / "source_complete.json"

    path = Path(repo) / source["path"]
    assert path.stat().st_size == source["bytes"]
    assert path.stat().st_mtime_ns == source["mtime_ns"]

    if done_path.exists():
        done = load_json(done_path)
        for item in done["files"]:
            verify_parquet(directory / item["name"], item["rows"])
        return done

    types = {
        "Time (EET)": pa.string(),
        "Ask": pa.float64(), "Bid": pa.float64(),
        "AskVolume": pa.float64(), "BidVolume": pa.float64()
    }
    reader = pacsv.open_csv(
        path,
        read_options=pacsv.ReadOptions(
            block_size=block_bytes, use_threads=False
        ),
        convert_options=pacsv.ConvertOptions(column_types=types)
    )
    assert reader.schema.names == list(types), f"Unexpected columns: {path}"

    writers, output_rows = {}, {}
    rows = locked = both_zero = 0
    first_ns = last_ns = previous_ns = None

    try:
        for batch in reader:
            count = batch.num_rows
            if not count:
                continue

            try:
                local = pd.DatetimeIndex(pd.to_datetime(
                    batch.column("Time (EET)").to_pylist(),
                    format="%Y.%m.%d %H:%M:%S.%f",
                    errors="raise"
                ))
                assert not local.hasnans
                local = local.tz_localize(
                    TZ, ambiguous="raise", nonexistent="raise"
                )
                ticks = local.tz_convert("UTC").as_unit("ns").asi8
            except Exception as error:
                batch.slice(0, min(count, 100)).to_pandas().to_csv(
                    directory / "timestamp_error_sample.csv", index=False
                )
                raise RuntimeError(
                    f"Timestamp review required in {path.name}; "
                    f"parsed records {rows + 1}–{rows + count}."
                ) from error

            values = {
                "bid": batch.column("Bid").to_numpy(zero_copy_only=False),
                "ask": batch.column("Ask").to_numpy(zero_copy_only=False),
                "bid_size": batch.column("BidVolume").to_numpy(zero_copy_only=False),
                "ask_size": batch.column("AskVolume").to_numpy(zero_copy_only=False)
            }
            invalid = np.zeros(count, dtype=bool)
            for array in values.values():
                invalid |= ~np.isfinite(array)
            invalid |= values["bid"] <= 0
            invalid |= values["ask"] <= 0
            invalid |= values["ask"] < values["bid"]
            invalid |= values["bid_size"] < 0
            invalid |= values["ask_size"] < 0

            if invalid.any():
                positions = np.flatnonzero(invalid)[:50]
                preview = batch.take(pa.array(positions)).to_pandas()
                preview["source_record"] = rows + positions + 1
                preview.to_csv(directory / "invalid_quote_examples.csv", index=False)
                raise RuntimeError(
                    f"Invalid quotes in {path.name}; review saved examples."
                )

            backward = np.any(ticks[1:] < ticks[:-1])
            backward |= previous_ns is not None and ticks[0] < previous_ns
            if backward:
                raise RuntimeError(f"Backward UTC timestamps require review: {path.name}")

            table = pa.table({
                "ts_ns": pa.array(ticks, type=pa.int64()),
                **{name: pa.array(array, type=pa.float64())
                   for name, array in values.items()},
                "source_id": pa.array(
                    np.full(count, source["source_id"], dtype=np.int32)
                ),
                "source_record": pa.array(
                    np.arange(rows + 1, rows + count + 1, dtype=np.int64)
                )
            })

            labels = np.asarray(local.strftime("%Y-%m"))
            for label in np.unique(labels):
                positions = np.flatnonzero(labels == label)
                if label not in writers:
                    writers[label] = pq.ParquetWriter(
                        directory / f"{label}.parquet",
                        table.schema,
                        compression="zstd"
                    )
                    output_rows[label] = 0
                writers[label].write_table(table.take(pa.array(positions)))
                output_rows[label] += len(positions)

            locked += int(np.count_nonzero(values["ask"] == values["bid"]))
            both_zero += int(np.count_nonzero(
                (values["bid_size"] == 0) & (values["ask_size"] == 0)
            ))
            first_ns = int(ticks[0]) if first_ns is None else first_ns
            previous_ns = last_ns = int(ticks[-1])
            rows += count
    finally:
        for writer in writers.values():
            writer.close()
        reader.close()

    assert rows > 0, f"Empty input requires review: {path}"
    assert path.stat().st_size == source["bytes"], "Source changed during processing."
    assert path.stat().st_mtime_ns == source["mtime_ns"], "Source changed during processing."

    files = [
        {"name": f"{label}.parquet", "year_month": label, "rows": int(count)}
        for label, count in sorted(output_rows.items())
    ]
    assert sum(item["rows"] for item in files) == rows

    done = {
        **source, "rows": rows,
        "first_ns": first_ns, "last_ns": last_ns,
        "locked_rows": locked, "both_sizes_zero_rows": both_zero,
        "files": files
    }
    dump_json(done_path, done)
    return done
''')

# Shared helpers: reconcile one month of observed quotes.
# Identical parsed quotes are combined.
# Conflicting quotes stop processing before a completed output is accepted.

HELPER_PARTS.append(r'''
def merge_month(con, files, destination, previous):
    con.read_parquet(
        [str(path) for path in files], hive_partitioning=False
    ).create_view("raw_month", replace=True)

    raw_rows = con.execute("SELECT count(*) FROM raw_month").fetchone()[0]

    con.execute("""
        CREATE OR REPLACE TEMP TABLE unique_month AS
        SELECT
            ts_ns, bid, ask, bid_size, ask_size,
            count(*)::BIGINT AS source_copies,
            arg_min(
                source_id,
                struct_pack(s := source_id, r := source_record)
            ) AS representative_source_id,
            arg_min(
                source_record,
                struct_pack(s := source_id, r := source_record)
            ) AS representative_source_record
        FROM raw_month
        GROUP BY ts_ns, bid, ask, bid_size, ask_size
    """)

    conflicts = con.execute("""
        SELECT count(*) FROM (
            SELECT ts_ns FROM unique_month
            GROUP BY ts_ns HAVING count(*) > 1
        )
    """).fetchone()[0]

    if conflicts:
        evidence = Path(destination).with_suffix(".conflicts.parquet")
        write_query(con, """
            SELECT * FROM unique_month
            WHERE ts_ns IN (
                SELECT ts_ns FROM unique_month
                GROUP BY ts_ns HAVING count(*) > 1
            )
            ORDER BY ts_ns, representative_source_id
        """, evidence)
        raise RuntimeError(
            f"{conflicts} conflicting timestamps. Review {evidence}"
        )

    previous_ns = "NULL" if previous is None else str(previous["ts_ns"])
    segment_offset = 0 if previous is None else previous["data_segment_id"]

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE ordered_month AS
        SELECT *,
            lag(ts_ns, 1, {previous_ns}) OVER (ORDER BY ts_ns) AS a_ns
        FROM unique_month
    """)
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE prepared_month AS
        SELECT *,
            ts_ns - a_ns AS gap_ns,
            {segment_offset} + sum(
                CASE WHEN ts_ns - a_ns > {LIMIT_NS} THEN 1 ELSE 0 END
            ) OVER (
                ORDER BY ts_ns ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS data_segment_id
        FROM ordered_month
    """)

    write_query(
        con, "SELECT * FROM prepared_month ORDER BY ts_ns", destination
    )
    con.read_parquet(str(destination), hive_partitioning=False).create_view(
        "saved_observed", replace=True
    )

    fields = "ts_ns, bid, ask, bid_size, ask_size"
    differences = con.execute(f"""
        SELECT count(*) FROM (
            (SELECT {fields} FROM unique_month
             EXCEPT ALL SELECT {fields} FROM saved_observed)
            UNION ALL
            (SELECT {fields} FROM saved_observed
             EXCEPT ALL SELECT {fields} FROM unique_month)
        )
    """).fetchone()[0]
    assert differences == 0, "Saved observed values do not reconcile."

    result = record(con, """
        SELECT
            count(*) AS observed_rows,
            sum(source_copies)::BIGINT AS represented_source_rows,
            count_if(ask = bid) AS locked_rows,
            count_if(bid_size = 0 AND ask_size = 0) AS both_sizes_zero_rows,
            count_if(gap_ns > 0) AS positive_intervals,
            count_if(gap_ns <= 0) AS nonpositive_intervals
        FROM saved_observed
    """)
    assert result["represented_source_rows"] == raw_rows
    assert result["nonpositive_intervals"] == 0

    result.update({
        "path": str(destination),
        "raw_rows": raw_rows,
        "duplicate_copies_removed": raw_rows - result["observed_rows"],
        "first": record(con, "SELECT * FROM saved_observed ORDER BY ts_ns LIMIT 1"),
        "last": record(con, "SELECT * FROM saved_observed ORDER BY ts_ns DESC LIMIT 1")
    })
    return result
''')

# Shared helpers: create interpolation context and independent expected keys.
# Neighboring months provide real anchors at partition boundaries.

HELPER_PARTS.append(r'''
KEYS = [
    "ts_ns", "left_anchor_ns", "right_anchor_ns",
    "available_ns", "data_segment_id", "scope_id"
]
KEY_SCHEMA = pa.schema([(name, pa.int64()) for name in KEYS])

def prepare_context(con, meta, previous, following, windows):
    path = meta["path"]
    con.read_parquet(path, hive_partitioning=False).create_view(
        "cur_quotes", replace=True
    )
    boundary = [row for row in (previous, following) if row is not None]
    con.register(
        "real_boundary",
        pa.Table.from_pylist(boundary, schema=pq.read_schema(path))
    )
    con.execute("""
        CREATE OR REPLACE TEMP TABLE pairs_quotes AS
        SELECT *,
            lag(ts_ns) OVER w AS left_ns,
            lag(bid) OVER w AS left_bid,
            lag(ask) OVER w AS left_ask,
            lag(bid_size) OVER w AS left_bid_size,
            lag(ask_size) OVER w AS left_ask_size
        FROM (
            SELECT * FROM cur_quotes
            UNION ALL SELECT * FROM real_boundary
        )
        WINDOW w AS (ORDER BY ts_ns)
    """)

    bad_adjacency = con.execute("""
        SELECT count(*) FROM pairs_quotes
        WHERE left_ns IS NOT NULL AND a_ns IS DISTINCT FROM left_ns
    """).fetchone()[0]
    assert bad_adjacency == 0, "Cross-partition real adjacency mismatch."

    con.register("exclusions", pa.table({
        "start_ns": pa.array([x[0] for x in windows], type=pa.int64()),
        "end_ns": pa.array([x[1] for x in windows], type=pa.int64()),
        "scope_id": pa.array(
            np.arange(1, len(windows) + 1, dtype=np.int64)
        )
    }))

def scope_sql(query):
    return f"""
        SELECT q.*, coalesce(e.scope_id, 0)::BIGINT AS scope_id
        FROM ({query}) q
        ASOF LEFT JOIN exclusions e ON q.ts_ns >= e.start_ns
        WHERE e.end_ns IS NULL OR q.ts_ns >= e.end_ns
    """

def write_expected_keys(con, path, lower, upper, windows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    reader = con.execute(f"""
        SELECT left_ns, ts_ns, data_segment_id
        FROM pairs_quotes
        WHERE ts_ns - left_ns > {STEP_NS}
          AND ts_ns - left_ns <= {LIMIT_NS}
          AND left_ns < {upper} AND ts_ns > {lower}
    """).fetch_record_batch(10_000)

    with pq.ParquetWriter(path, KEY_SCHEMA, compression="zstd") as writer:
        for batch in reader:
            a = batch.column(0).to_numpy()
            b = batch.column(1).to_numpy()
            segment = batch.column(2).to_numpy()
            counts = ((b - a - 1) // STEP_NS).astype(np.int64)
            total = int(counts.sum())
            if not total:
                continue

            starts = np.cumsum(counts) - counts
            k = np.arange(total, dtype=np.int64) - np.repeat(starts, counts) + 1
            left = np.repeat(a, counts)
            right = np.repeat(b, counts)
            ticks = left + k * STEP_NS
            allowed, scope = allowed_and_scope(ticks, windows)
            keep = allowed & (ticks >= lower) & (ticks < upper)

            arrays = [
                ticks[keep], left[keep], right[keep], right[keep],
                np.repeat(segment, counts)[keep], scope[keep]
            ]
            writer.write_table(pa.Table.from_arrays(
                [pa.array(array, type=pa.int64()) for array in arrays],
                schema=KEY_SCHEMA
            ))
''')

In [ ]:
# Shared helpers: write and audit one analysis month.
# The audit reads the saved Parquet output.

HELPER_PARTS.append(r'''
def analysis_month(
    con, meta, previous, following, windows,
    destination, expected_path, previous_analysis, rebuild
):
    prepare_context(con, meta, previous, following, windows)
    lower, upper = month_bounds(meta["year_month"])

    real = """
        SELECT
            ts_ns, ts_ns AS available_ns,
            bid, ask, bid_size, ask_size,
            false AS is_interpolated,
            ts_ns AS left_anchor_ns, ts_ns AS right_anchor_ns,
            data_segment_id,
            representative_source_id, representative_source_record,
            source_copies
        FROM cur_quotes
    """

    interpolated_values = ", ".join(
        f"left_{name} + ({name} - left_{name}) * "
        f"((k * {STEP_NS})::DOUBLE / (ts_ns - left_ns)::DOUBLE) AS {name}"
        for name in VALUES
    )
    synthetic = f"""
        SELECT
            left_ns + k * {STEP_NS} AS ts_ns,
            ts_ns AS available_ns,
            {interpolated_values},
            true AS is_interpolated,
            left_ns AS left_anchor_ns,
            ts_ns AS right_anchor_ns,
            data_segment_id,
            NULL::INTEGER AS representative_source_id,
            NULL::BIGINT AS representative_source_record,
            0::BIGINT AS source_copies
        FROM pairs_quotes,
        LATERAL range(
            1::BIGINT,
            ((ts_ns - left_ns - 1) // {STEP_NS} + 1)::BIGINT
        ) generated(k)
        WHERE ts_ns - left_ns > {STEP_NS}
          AND ts_ns - left_ns <= {LIMIT_NS}
    """

    candidates = f"""
        SELECT * FROM ({real} UNION ALL {synthetic})
        WHERE ts_ns >= {lower} AND ts_ns < {upper}
    """
    scoped = scope_sql(candidates)

    if rebuild:
        write_query(con, f"""
            SELECT *,
                ask = bid AS locked_quote,
                bid_size = 0 AND ask_size = 0 AS both_sizes_zero
            FROM ({scoped})
            ORDER BY ts_ns
        """, destination)

    write_expected_keys(con, expected_path, lower, upper, windows)
    con.read_parquet(str(destination), hive_partitioning=False).create_view(
        "actual_quotes", replace=True
    )
    con.read_parquet(str(expected_path), hive_partitioning=False).create_view(
        "expected_keys", replace=True
    )

    key_sql = ", ".join(KEYS)
    checks = {}

    checks["missing_required_fill_points"] = con.execute(f"""
        SELECT count(*) FROM (
            SELECT {key_sql} FROM expected_keys
            EXCEPT ALL
            SELECT {key_sql} FROM actual_quotes WHERE is_interpolated
        )
    """).fetchone()[0]

    checks["unexpected_synthetic_points"] = con.execute(f"""
        SELECT count(*) FROM (
            SELECT {key_sql} FROM actual_quotes WHERE is_interpolated
            EXCEPT ALL SELECT {key_sql} FROM expected_keys
        )
    """).fetchone()[0]

    checks["duplicate_analysis_timestamps"] = con.execute("""
        SELECT count(*) - count(DISTINCT ts_ns) FROM actual_quotes
    """).fetchone()[0]

    checks["rows_inside_exclusions"] = con.execute("""
        SELECT count(*)
        FROM actual_quotes q
        ASOF LEFT JOIN exclusions e ON q.ts_ns >= e.start_ns
        WHERE q.ts_ns < e.end_ns
    """).fetchone()[0]

    checks["invalid_analysis_values"] = con.execute("""
        SELECT count(*) FROM actual_quotes
        WHERE NOT isfinite(bid) OR NOT isfinite(ask)
           OR NOT isfinite(bid_size) OR NOT isfinite(ask_size)
           OR bid <= 0 OR ask < bid OR bid_size < 0 OR ask_size < 0
           OR available_ns < ts_ns
           OR (NOT is_interpolated AND available_ns <> ts_ns)
    """).fetchone()[0]

    expected_real = scope_sql(real)
    columns = [
        "ts_ns", "available_ns", *VALUES, "is_interpolated",
        "left_anchor_ns", "right_anchor_ns", "data_segment_id",
        "representative_source_id", "representative_source_record",
        "source_copies", "scope_id"
    ]
    columns_sql = ", ".join(columns)
    checks["changed_or_missing_real_rows"] = con.execute(f"""
        SELECT count(*) FROM (
            (SELECT {columns_sql} FROM ({expected_real})
             EXCEPT ALL
             SELECT {columns_sql} FROM actual_quotes WHERE NOT is_interpolated)
            UNION ALL
            (SELECT {columns_sql} FROM actual_quotes WHERE NOT is_interpolated
             EXCEPT ALL
             SELECT {columns_sql} FROM ({expected_real}))
        )
    """).fetchone()[0]

    value_errors = []
    for name in VALUES:
        expected_value = (
            f"p.left_{name} + (p.{name} - p.left_{name}) * "
            "((q.ts_ns - p.left_ns)::DOUBLE / "
            "(p.ts_ns - p.left_ns)::DOUBLE)"
        )
        value_errors.append(
            f"abs(q.{name} - ({expected_value})) > "
            f"1e-12 * greatest(1.0, abs({expected_value}))"
        )

    checks["interpolation_value_errors"] = con.execute(f"""
        SELECT count(*)
        FROM actual_quotes q
        JOIN pairs_quotes p
          ON q.right_anchor_ns = p.ts_ns
         AND q.left_anchor_ns = p.left_ns
        WHERE q.is_interpolated AND (
            q.available_ns <> p.ts_ns
            OR p.ts_ns - p.left_ns > {LIMIT_NS}
            OR q.ts_ns <= p.left_ns OR q.ts_ns >= p.ts_ns
            OR {" OR ".join(value_errors)}
        )
    """).fetchone()[0]

    boundary_schema = pa.schema([
        ("ts_ns", pa.int64()),
        ("data_segment_id", pa.int64()),
        ("scope_id", pa.int64())
    ])
    con.register("analysis_boundary", pa.Table.from_pylist(
        [] if previous_analysis is None else [previous_analysis],
        schema=boundary_schema
    ))
    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW analysis_spacings AS
        WITH combined AS (
            SELECT ts_ns, data_segment_id, scope_id FROM actual_quotes
            UNION ALL SELECT * FROM analysis_boundary
        ), ordered AS (
            SELECT *,
                lag(ts_ns) OVER w AS previous_ns,
                lag(data_segment_id) OVER w AS previous_segment,
                lag(scope_id) OVER w AS previous_scope
            FROM combined WINDOW w AS (ORDER BY ts_ns)
        )
        SELECT ts_ns - previous_ns AS gap_ns
        FROM ordered
        WHERE ts_ns >= {lower}
          AND data_segment_id = previous_segment
          AND scope_id = previous_scope
    """)

    checks["unresolved_short_gap_instances"] = con.execute(f"""
        SELECT count(*) FROM analysis_spacings
        WHERE gap_ns > {STEP_NS} AND gap_ns <= {LIMIT_NS}
    """).fetchone()[0]
    checks["long_spacing_inside_continuous_segment"] = con.execute(f"""
        SELECT count(*) FROM analysis_spacings WHERE gap_ns > {LIMIT_NS}
    """).fetchone()[0]
    checks["nonpositive_analysis_spacing"] = con.execute("""
        SELECT count(*) FROM analysis_spacings WHERE gap_ns <= 0
    """).fetchone()[0]

    result = record(con, """
        SELECT
            count(*) AS analysis_rows,
            count_if(is_interpolated) AS synthetic_rows,
            count_if(NOT is_interpolated) AS retained_real_rows
        FROM actual_quotes
    """)
    result.update({
        "year_month": meta["year_month"],
        "path": str(destination),
        "expected_fill_points": con.execute(
            "SELECT count(*) FROM expected_keys"
        ).fetchone()[0],
        "checks": checks,
        "raw_interval_histogram": histogram(con, "cur_quotes"),
        "analysis_spacing_histogram": histogram(con, "analysis_spacings"),
        "last": record(con, """
            SELECT ts_ns, data_segment_id, scope_id
            FROM actual_quotes ORDER BY ts_ns DESC LIMIT 1
        """)
    })

    assert all(value == 0 for value in checks.values()), checks
    assert result["synthetic_rows"] == result["expected_fill_points"]
    verify_parquet(destination, result["analysis_rows"])
    return result
''')

In [ ]:
# Correct the shared helper for DuckDB 2.0.
#
# - Finish small query results before reusing the connection.
# - Keep large results streamed through the Arrow reader.
# - Close the reader after use.
# - Retain the earlier counter and integer-type fixes.
# - Preserve original CSVs and earlier run folders.

helper_source = "\n".join(HELPER_PARTS)
helper_source = helper_source.replace("\r\n", "\n").replace("\r", "\n")

assert helper_source.count("def merge_month(") == 1, (
    "Rerun Cells 4–9 in order to reset the helper source."
)
assert helper_source.count("def analysis_month(") == 1

# Counts must return zero for empty or all-null groups.
conditions = [
    "ask = bid",
    "bid_size = 0 AND ask_size = 0",
    "gap_ns > 0",
    "gap_ns <= 0",
    "is_interpolated",
    "NOT is_interpolated"
]

for condition in conditions:
    old = f"count_if({condition})"
    new = f"count(*) FILTER (WHERE {condition})"
    assert old in helper_source or new in helper_source
    helper_source = helper_source.replace(old, new)

# Keep saved segment identifiers as 64-bit integers.
segment_expression = """{segment_offset} + sum(
                CASE WHEN ts_ns - a_ns > {LIMIT_NS} THEN 1 ELSE 0 END
            ) OVER (
                ORDER BY ts_ns ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )"""

old_segment = segment_expression + " AS data_segment_id"
new_segment = (
    "CAST((" + segment_expression +
    ") AS BIGINT) AS data_segment_id"
)

assert old_segment in helper_source or new_segment in helper_source
helper_source = helper_source.replace(old_segment, new_segment)

# These calls retrieve single-row scalar results, not market history.
helper_source = helper_source.replace(
    ".fetchone()[0]",
    ".fetchall()[0][0]"
)

replacement_functions = {
    "write_query": r'''
def write_query(con, query, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial.parquet")
    temporary.unlink(missing_ok=True)

    con.execute(
        f"COPY ({query}) TO {literal(temporary)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD)"
    )
    con.complete()
    temporary.replace(path)
''',

    "record": r'''
def record(con, query):
    cursor = con.execute(query)
    names = [item[0] for item in cursor.description]
    rows = cursor.fetchall()

    assert len(rows) <= 1, "record() requires a single-row query."
    return None if not rows else dict(zip(names, rows[0]))
''',

    "write_expected_keys": r'''
def write_expected_keys(con, path, lower, upper, windows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    query = f"""
        SELECT left_ns, ts_ns, data_segment_id
        FROM pairs_quotes
        WHERE ts_ns - left_ns > {STEP_NS}
          AND ts_ns - left_ns <= {LIMIT_NS}
          AND left_ns < {upper}
          AND ts_ns > {lower}
    """

    reader = con.sql(query).to_arrow_reader(batch_size=10_000)

    try:
        with pq.ParquetWriter(
            path, KEY_SCHEMA, compression="zstd"
        ) as writer:
            for batch in reader:
                a = batch.column(0).to_numpy()
                b = batch.column(1).to_numpy()
                segment = batch.column(2).to_numpy()

                counts = ((b - a - 1) // STEP_NS).astype(np.int64)
                total = int(counts.sum())
                if not total:
                    continue

                starts = np.cumsum(counts) - counts
                k = (
                    np.arange(total, dtype=np.int64)
                    - np.repeat(starts, counts)
                    + 1
                )

                left = np.repeat(a, counts)
                right = np.repeat(b, counts)
                ticks = left + k * STEP_NS

                allowed, scope = allowed_and_scope(ticks, windows)
                keep = allowed & (ticks >= lower) & (ticks < upper)

                arrays = [
                    ticks[keep],
                    left[keep],
                    right[keep],
                    right[keep],
                    np.repeat(segment, counts)[keep],
                    scope[keep]
                ]

                writer.write_table(pa.Table.from_arrays(
                    [
                        pa.array(array, type=pa.int64())
                        for array in arrays
                    ],
                    schema=KEY_SCHEMA
                ))
    finally:
        reader.close()
'''
}

# Replace only the named functions inside the generated helper.
for name, replacement in replacement_functions.items():
    signature = f"def {name}("
    assert helper_source.count(signature) == 1

    start = helper_source.index(signature)
    end = helper_source.find("\ndef ", start + len(signature))
    assert end >= 0, f"Cannot locate the end of {name}."

    helper_source = (
        helper_source[:start]
        + replacement.strip()
        + "\n\n"
        + helper_source[end + 1:]
    )

assert ".fetchone(" not in helper_source
assert ".fetch_record_batch(" not in helper_source
assert "count_if(" not in helper_source

HELPER.parent.mkdir(parents=True, exist_ok=True)
HELPER.write_text(
    helper_source,
    encoding="utf-8",
    newline="\n"
)

spec = importlib.util.spec_from_file_location("fx_etl_helpers", HELPER)
fx = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fx)

fx.LIMIT_NS = MAX_GAP_SECONDS * 1_000_000_000
fx.STEP_NS = INTERPOLATION_STEP_SECONDS * 1_000_000_000

# Use a new configuration snapshot for this corrected helper.
if RUN_ID in {"etl_01_v1", "etl_01_v2"}:
    RUN_ID = "etl_01_v3"

RUN = DATA / "processed" / RUN_ID
REPORT = REPO / "reports" / "tables" / "etl" / RUN_ID
FIGURES = REPO / "reports" / "figures" / "etl" / RUN_ID

FINAL_AUDIT_PASSED = False

print("DuckDB 2.0 helper corrections loaded.")
print("Run selected:", RUN_ID)
print("Next: run Cell 11, then the unchanged Cell 12.")

In [ ]:
# Inventory and run protection.
# Use a new run for the corrected helper.
# Preserve the earlier run and its configuration.
# A later custom RUN_ID in Cell 2 still takes precedence.

FINAL_AUDIT_PASSED = False

if RUN_ID == "etl_01_v1":
    RUN_ID = "etl_01_v2"

RUN = DATA / "processed" / RUN_ID
REPORT = REPO / "reports" / "tables" / "etl" / RUN_ID
FIGURES = REPO / "reports" / "figures" / "etl" / RUN_ID

original_inventory = pd.read_csv(
    AUDIT / "input_inventory.csv",
    dtype={
        "source_file": str,
        "bytes": "int64",
        "mtime_ns": "int64"
    }
)
original_audit = fx.load_json(AUDIT / "audit_summary.json")

original_paths = [
    REPO / name for name in original_inventory["source_file"]
]
assert all(path.is_file() for path in original_paths)

discovered_originals = set(DATA.glob("EURUSD_Ticks_*.csv"))
assert discovered_originals == set(original_paths), (
    "Original-file inventory changed. Review it before continuing."
)

new_paths = sorted(NEW_EXPORTS.rglob("EURUSD_Ticks_*.csv"))
assert new_paths, "No new exports found."

sources = []
for source_id, path in enumerate(original_paths + new_paths):
    stat = path.stat()
    sources.append({
        "source_id": source_id,
        "path": path.relative_to(REPO).as_posix(),
        "bytes": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
        "original": source_id < len(original_paths)
    })

for index, old in enumerate(original_inventory.to_dict("records")):
    current = sources[index]
    assert current["bytes"] == old["bytes"], (
        f"An original CSV changed size: {current['path']}"
    )
    assert current["mtime_ns"] == old["mtime_ns"], (
        f"An original CSV modification time changed: {current['path']}"
    )

buckets = pd.read_csv(AUDIT / "gap_bucket_definitions.csv")
assert len(buckets) == 56

fx.BUCKETS = [
    (
        int(row["bucket_index"]),
        int(round(row["upper_seconds_inclusive"] * 1e9))
        if np.isfinite(row["upper_seconds_inclusive"])
        else int(np.iinfo(np.int64).max)
    )
    for row in buckets.to_dict("records")
]

verified_closures = AUDIT / "verified_quote_closures_used.csv"
assert verified_closures.is_file()

configuration = {
    "run_id": RUN_ID,
    "audit_run": AUDIT_RUN,
    "versions": VERSIONS,
    "sources": sources,
    "source_timezone": SOURCE_TZ,
    "display_timezone": RESEARCH_TZ,
    "gap_limit_seconds": MAX_GAP_SECONDS,
    "step_seconds": INTERPOLATION_STEP_SECONDS,
    "csv_block_mib": CSV_BLOCK_MIB,
    "helper_sha256": hashlib.sha256(
        HELPER.read_bytes()
    ).hexdigest(),
    "bucket_sha256": hashlib.sha256(
        (AUDIT / "gap_bucket_definitions.csv").read_bytes()
    ).hexdigest(),
    "verified_closure_sha256": hashlib.sha256(
        verified_closures.read_bytes()
    ).hexdigest(),
    "daily_exclusions":
        "16:00–19:00 America/New_York, Sunday–Friday",
    "weekend":
        "Friday 17:00–Sunday 17:00 America/New_York",
    "january_1":
        "full Europe/Athens day",
    "boundary_anchors":
        "real anchors may lie inside excluded hours"
}

FINGERPRINT = hashlib.sha256(
    json.dumps(configuration, sort_keys=True).encode()
).hexdigest()

free_gib = shutil.disk_usage(REPO).free / 2**30
print(f"Free disk: {free_gib:,.1f} GiB")

assert free_gib >= MIN_FREE_GIB, (
    "Below the configured disk preflight minimum. "
    "Review available disk and MIN_FREE_GIB."
)

for directory in (RUN, REPORT, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

config_path = RUN / "configuration.json"

if config_path.exists():
    saved_configuration = fx.load_json(config_path)

    changed_fields = [
        key
        for key in sorted(
            set(saved_configuration) | set(configuration)
        )
        if saved_configuration.get(key) != configuration.get(key)
    ]

    assert not changed_fields, (
        f"Configuration differs in: {', '.join(changed_fields)}. "
        "Choose a new RUN_ID in Cell 2 and rerun Cells 2–12."
    )
else:
    fx.dump_json(config_path, configuration)

fx.dump_json(RUN / "run_state.json", {
    "status": "running",
    "fingerprint": FINGERPRINT,
    "human_review": "pending"
})

pd.DataFrame(sources).to_csv(
    REPORT / "input_inventory.csv",
    index=False
)
buckets.to_csv(
    REPORT / "gap_bucket_definitions.csv",
    index=False
)

print("Active run:", RUN_ID)
print("Configuration accepted.")
display(pd.DataFrame(sources)[[
    "source_id", "original", "path", "bytes"
]])

In [ ]:
# Small synthetic checks before the full-history operation.
#
# These use the same merge, interpolation and saved-output audit functions.
# They cover:
# - Exactly 10 seconds.
# - Exactly 20 seconds.
# - Exactly 300 seconds.
# - 300 seconds plus one nanosecond.
# - A month boundary.
# - Exclusion start/end boundaries.
# - A long gap whose allowed remainder is shorter than five minutes.
# - Conflicting quotes.

with tempfile.TemporaryDirectory(prefix="etl_checks_", dir=RUN) as temporary:
    test_dir = Path(temporary)
    origin = pd.Timestamp("2024-01-31 23:59:40", tz=SOURCE_TZ).value
    offsets = np.array([
        0,
        40_000_000_000,
        50_000_000_000,
        70_000_000_000,
        370_000_000_000,
        670_000_000_001
    ], dtype=np.int64)
    ticks = origin + offsets
    bid = 1.10 + np.arange(len(ticks)) * 0.0001

    toy = pa.table({
        "ts_ns": ticks,
        "bid": bid,
        "ask": bid + 0.0002,
        "bid_size": 10.0 + np.arange(len(ticks)),
        "ask_size": 12.0 + np.arange(len(ticks)),
        "source_id": np.zeros(len(ticks), dtype=np.int32),
        "source_record": np.arange(1, len(ticks) + 1, dtype=np.int64)
    })
    labels = pd.to_datetime(
        ticks, unit="ns", utc=True
    ).tz_convert(SOURCE_TZ).strftime("%Y-%m")

    toy_windows = fx.merge_windows([
        (origin + 20_000_000_000, origin + 30_000_000_000),
        (origin + 370_000_000_000, origin + 430_000_000_000)
    ])

    with duckdb.connect() as test_con:
        test_meta = []
        previous = None
        for label in sorted(set(labels)):
            raw_path = test_dir / f"raw_{label}.parquet"
            pq.write_table(
                toy.take(pa.array(np.flatnonzero(labels == label))),
                raw_path
            )
            result = fx.merge_month(
                test_con, [raw_path],
                test_dir / f"observed_{label}.parquet",
                previous
            )
            result["year_month"] = label
            test_meta.append(result)
            previous = result["last"]

        previous_analysis = None
        test_results = []
        for index, meta in enumerate(test_meta):
            result = fx.analysis_month(
                test_con,
                meta,
                test_meta[index - 1]["last"] if index else None,
                test_meta[index + 1]["first"]
                if index + 1 < len(test_meta) else None,
                toy_windows,
                test_dir / f"analysis_{meta['year_month']}.parquet",
                test_dir / f"expected_{meta['year_month']}.parquet",
                previous_analysis,
                rebuild=True
            )
            test_results.append(result)
            previous_analysis = result["last"] or previous_analysis

        # 33 possible synthetic points; one lies inside an exclusion.
        assert sum(row["synthetic_rows"] for row in test_results) == 32
        assert test_meta[-1]["last"]["data_segment_id"] == 1

        pieces = list(fx.allowed_pieces(
            origin + 370_000_000_000,
            origin + 670_000_000_001,
            toy_windows
        ))
        assert len(pieces) == 1
        assert pieces[0][1] - pieces[0][0] == 240_000_000_001

        allowed, scope = fx.allowed_and_scope(
            np.array([origin + 20_000_000_000, origin + 30_000_000_000]),
            toy_windows
        )
        assert allowed.tolist() == [False, True]

        conflicting = pa.table({
            **toy.slice(0, 1).to_pydict(),
            "bid": [float(bid[0] - 0.0001)]
        })
        conflict_path = test_dir / "conflicting.parquet"
        pq.write_table(conflicting, conflict_path)

        try:
            fx.merge_month(
                test_con,
                [test_dir / "raw_2024-01.parquet", conflict_path],
                test_dir / "must_not_complete.parquet",
                None
            )
        except RuntimeError as error:
            assert "conflicting timestamps" in str(error)
        else:
            raise AssertionError("Conflicting quotes were not detected.")

print("Synthetic checks passed.")

In [ ]:
# EXPENSIVE: full-history CSV scan.
# Every original and new export is included.
# Completed source checkpoints are reused on an unchanged run.
# Keep the staging files: they preserve source-level provenance.

source_results = []

for source in sources:
    directory = RUN / "_staging" / f"source_{source['source_id']:03d}"
    print("Scanning/reusing:", source["path"], flush=True)

    result = fx.stage_source(
        REPO, source, directory,
        CSV_BLOCK_MIB * 1024 * 1024
    )
    source_results.append(result)

assert len(source_results) == len(sources)

original_rows = sum(
    result["rows"] for result in source_results if result["original"]
)
assert original_rows == original_audit["source_records"], (
    "Original CSV row totals differ from Notebook 00."
)

source_table = pd.DataFrame([
    {
        "source_id": result["source_id"],
        "original": result["original"],
        "path": result["path"],
        "rows": result["rows"],
        "first_timestamp_athens": fx.athens(result["first_ns"]),
        "last_timestamp_athens": fx.athens(result["last_ns"]),
        "locked_rows": result["locked_rows"],
        "both_sizes_zero_rows": result["both_sizes_zero_rows"]
    }
    for result in source_results
])

source_table.to_csv(REPORT / "source_summary.csv", index=False)
display(source_table)

In [ ]:
# EXPENSIVE: merge the full observed history, one month at a time.
# Monthly processing is internal; this remains one full-history notebook.
# A conflicting timestamp stops here and produces a Parquet evidence file.

con = duckdb.connect(str(RUN / "_work.duckdb"))
con.execute(f"SET memory_limit = {fx.literal(DUCKDB_MEMORY_LIMIT)}")
con.execute(f"SET threads = {int(DUCKDB_THREADS)}")
con.execute(
    f"SET temp_directory = {fx.literal(RUN / '_duckdb_spill')}"
)

month_inputs = {}
for result in source_results:
    directory = RUN / "_staging" / f"source_{result['source_id']:03d}"
    for item in result["files"]:
        month_inputs.setdefault(item["year_month"], []).append(
            directory / item["name"]
        )

observed_meta = []
previous_real = None

for label, files in sorted(month_inputs.items()):
    destination = RUN / "observed" / f"{label}.parquet"
    done_path = RUN / "_checkpoints" / f"observed_{label}.json"
    expected_raw_rows = sum(
        pq.ParquetFile(path).metadata.num_rows for path in files
    )

    print("Reconciling/reusing observed month:", label, flush=True)

    if done_path.exists():
        result = fx.load_json(done_path)
        fx.verify_parquet(destination, result["observed_rows"])
        assert result["raw_rows"] == expected_raw_rows
        expected_previous = None if previous_real is None else previous_real["ts_ns"]
        assert result["first"]["a_ns"] == expected_previous
    else:
        result = fx.merge_month(con, files, destination, previous_real)
        result["year_month"] = label
        assert result["raw_rows"] == expected_raw_rows
        fx.dump_json(done_path, result)

    observed_meta.append(result)
    previous_real = result["last"]

assert sum(row["raw_rows"] for row in observed_meta) == sum(
    row["rows"] for row in source_results
)
assert sum(row["positive_intervals"] for row in observed_meta) == (
    sum(row["observed_rows"] for row in observed_meta) - 1
)

print("All observed months reconciled.")

In [ ]:
# Build the agreed analysis exclusions.
# Times are localized using named zones, including DST changes.
# Exclusions are half-open: start excluded, end allowed.

FIRST_NS = observed_meta[0]["first"]["ts_ns"]
LAST_NS = observed_meta[-1]["last"]["ts_ns"]

windows, window_details = fx.make_windows(
    FIRST_NS, LAST_NS, verified_closures
)

window_table = pd.DataFrame(window_details)
window_table["start_athens"] = window_table["start_ns"].map(fx.athens)
window_table["end_athens"] = window_table["end_ns"].map(fx.athens)
window_table.to_csv(REPORT / "exclusion_definitions.csv", index=False)

fx.dump_json(RUN / "merged_exclusion_windows.json", windows)

# Show the March clock difference without hard-coding UTC offsets.
dst_table = pd.DataFrame([
    {
        "new_york_boundary": str(
            pd.Timestamp(f"{day} 17:00", tz="America/New_York")
        ),
        "same_instant_athens": str(
            pd.Timestamp(
                f"{day} 17:00", tz="America/New_York"
            ).tz_convert(SOURCE_TZ)
        )
    }
    for day in ["2026-03-01", "2026-03-08", "2026-03-15", "2026-03-22", "2026-03-29"]
])

display(dst_table)
print("Merged exclusion windows:", len(windows))

In [ ]:
# EXPENSIVE: write and audit the full analysis history.
#
# The saved output contains retained real quotes plus synthetic points.
# It remains irregular: this is not a regular candle or sampling grid.
#
# The audit independently regenerates required point keys with NumPy.
# Missing and unexpected points are checked against saved Parquet.
# All audit error counts must be zero.

analysis_results = []
previous_analysis = None

for index, meta in enumerate(observed_meta):
    label = meta["year_month"]
    destination = RUN / "analysis" / f"{label}.parquet"
    expected_path = RUN / "_expected_fill_keys" / f"{label}.parquet"
    done_path = RUN / "_checkpoints" / f"analysis_{label}.json"

    rebuild = not done_path.exists()
    if not rebuild:
        old = fx.load_json(done_path)
        fx.verify_parquet(destination, old["analysis_rows"])

    print("Building/auditing analysis month:", label, flush=True)

    result = fx.analysis_month(
        con,
        meta,
        observed_meta[index - 1]["last"] if index else None,
        observed_meta[index + 1]["first"]
        if index + 1 < len(observed_meta) else None,
        windows,
        destination,
        expected_path,
        previous_analysis,
        rebuild=rebuild
    )

    fx.dump_json(done_path, result)
    analysis_results.append(result)
    previous_analysis = result["last"] or previous_analysis

assert len(analysis_results) == len(observed_meta)

checks_table = pd.DataFrame([
    {"year_month": result["year_month"], **result["checks"]}
    for result in analysis_results
])
checks_table.to_csv(REPORT / "final_checks_by_month.csv", index=False)

print("Final audit totals:")
display(checks_table.drop(columns="year_month").sum().to_frame("instances"))

assert (checks_table.drop(columns="year_month") == 0).all().all()

In [ ]:
# Dead-zone report.
#
# Raw intervals over 300 seconds retain their original classification.
# Only their time outside exclusions is counted as analysis dead-zone time.
# A weekend or January 1 interval fully covered by exclusions is not an outage.
# A dead interval spanning months is split for monthly duration reporting.
#
# These records show an absence of observed quotes.
# They do not establish the cause of that absence.

observed_paths = [row["path"] for row in observed_meta]
con.read_parquet(
    observed_paths, hive_partitioning=False
).create_view("all_observed", replace=True)

raw_long_path = REPORT / "long_real_intervals_all_hours.parquet"
fx.write_query(con, f"""
    SELECT
        a_ns,
        ts_ns AS b_ns,
        gap_ns AS raw_gap_ns
    FROM all_observed
    WHERE gap_ns > {fx.LIMIT_NS}
    ORDER BY a_ns
""", raw_long_path)

piece_schema = pa.schema([
    ("parent_id", pa.int64()),
    ("a_ns", pa.int64()),
    ("b_ns", pa.int64()),
    ("raw_gap_ns", pa.int64()),
    ("piece_start_ns", pa.int64()),
    ("piece_end_ns", pa.int64()),
    ("piece_duration_ns", pa.int64()),
    ("year_month", pa.string())
])

dead_path = REPORT / "dead_zone_pieces.parquet"

with pq.ParquetWriter(dead_path, piece_schema, compression="zstd") as writer:
    reader = pq.ParquetFile(raw_long_path).iter_batches(batch_size=10_000)
    for batch in reader:
        output = []
        for row in batch.to_pylist():
            for start, end in fx.allowed_pieces(
                row["a_ns"], row["b_ns"], windows
            ):
                for label, left, right in fx.split_by_month(start, end):
                    output.append({
                        "parent_id": row["b_ns"],
                        **row,
                        "piece_start_ns": left,
                        "piece_end_ns": right,
                        "piece_duration_ns": right - left,
                        "year_month": label
                    })
        if output:
            writer.write_table(
                pa.Table.from_pylist(output, schema=piece_schema)
            )

con.read_parquet(
    str(dead_path), hive_partitioning=False
).create_view("dead_pieces", replace=True)

assert con.execute(f"""
    SELECT count(*) FROM dead_pieces
    WHERE raw_gap_ns <= {fx.LIMIT_NS}
       OR piece_duration_ns <= 0
       OR piece_start_ns < a_ns
       OR piece_end_ns > b_ns
""").fetchone()[0] == 0

dead_monthly = con.execute("""
    SELECT
        year_month,
        count(DISTINCT parent_id) AS dead_parent_intervals,
        count(*) AS allowed_pieces,
        sum(piece_duration_ns) / 3.6e12 AS dead_hours,
        max(piece_duration_ns) / 6e10 AS longest_allowed_piece_minutes,
        max(raw_gap_ns) / 6e10 AS longest_raw_parent_minutes
    FROM dead_pieces
    GROUP BY year_month ORDER BY year_month
""").df()

dead_monthly.to_csv(REPORT / "dead_zone_monthly.csv", index=False)

largest_dead = con.execute("""
    SELECT * FROM dead_pieces
    ORDER BY piece_duration_ns DESC LIMIT 20
""").df()

for column in ["a_ns", "b_ns", "piece_start_ns", "piece_end_ns"]:
    largest_dead[column.replace("_ns", "_athens")] = (
        largest_dead[column].map(fx.athens)
    )

largest_dead["raw_gap_minutes"] = largest_dead["raw_gap_ns"] / 6e10
largest_dead["allowed_piece_minutes"] = (
    largest_dead["piece_duration_ns"] / 6e10
)
largest_dead.to_csv(REPORT / "largest_dead_zone_pieces.csv", index=False)

display(dead_monthly)
display(largest_dead[[
    "year_month", "a_athens", "b_athens",
    "piece_start_athens", "piece_end_athens",
    "raw_gap_minutes", "allowed_piece_minutes"
]])

In [ ]:
# Reconciliation and the agreed 56 gap buckets.
#
# These two distributions have different scopes:
# - Real intervals: all observed hours, including closures.
# - Analysis spacing: retained rows within the same continuity segment.
#
# The second distribution must have no spacing above 10 seconds.
# Ordinary positive intervals below 10 seconds remain valid.

monthly_reconciliation = pd.DataFrame([
    {
        "year_month": observed["year_month"],
        "source_rows": observed["raw_rows"],
        "observed_rows": observed["observed_rows"],
        "duplicate_copies_removed": observed["duplicate_copies_removed"],
        "locked_observed_rows": observed["locked_rows"],
        "both_sizes_zero_observed_rows": observed["both_sizes_zero_rows"],
        "retained_real_rows": analysis["retained_real_rows"],
        "synthetic_rows": analysis["synthetic_rows"],
        "analysis_rows": analysis["analysis_rows"],
        "missing_required_fill_points":
            analysis["checks"]["missing_required_fill_points"],
        "unresolved_short_gap_instances":
            analysis["checks"]["unresolved_short_gap_instances"]
    }
    for observed, analysis in zip(observed_meta, analysis_results)
])

monthly_reconciliation.to_csv(
    REPORT / "monthly_reconciliation.csv", index=False
)

raw_counts = np.asarray([
    row["raw_interval_histogram"] for row in analysis_results
], dtype=np.int64).sum(axis=0)

analysis_counts = np.asarray([
    row["analysis_spacing_histogram"] for row in analysis_results
], dtype=np.int64).sum(axis=0)

assert int(raw_counts.sum()) == (
    sum(row["observed_rows"] for row in observed_meta) - 1
)

bucket_report = buckets.copy()
bucket_report["real_intervals_all_hours"] = raw_counts
bucket_report["analysis_spacing_within_segments"] = analysis_counts

assert bucket_report.loc[
    bucket_report["lower_seconds_exclusive"] >= 10,
    "analysis_spacing_within_segments"
].sum() == 0

bucket_report.to_csv(REPORT / "gap_bucket_distribution.csv", index=False)

display(monthly_reconciliation)
display(bucket_report)

In [ ]:
# Final execution summary.
# A completed machine audit does not mean human review is complete.
#
# "Zero unresolved short gaps" means:
# - No required A+k*10-second fill points are missing.
# - No spacing above 10 seconds remains inside an allowed continuous segment.
#
# It does not mean real ticks arrive continuously.
# Coverage before the first quote and after the last quote is not inferred.

for source in sources:
    path = REPO / source["path"]
    assert path.stat().st_size == source["bytes"]
    assert path.stat().st_mtime_ns == source["mtime_ns"]

dead_totals = fx.record(con, """
    SELECT
        count(DISTINCT parent_id) AS dead_parent_intervals,
        count(*) AS dead_zone_pieces,
        coalesce(sum(piece_duration_ns), 0) / 3.6e12 AS dead_hours
    FROM dead_pieces
""")

check_totals = {
    column: int(checks_table[column].sum())
    for column in checks_table.columns if column != "year_month"
}
assert all(value == 0 for value in check_totals.values())

total_source_rows = sum(row["rows"] for row in source_results)
total_observed_rows = sum(row["observed_rows"] for row in observed_meta)
total_duplicates = sum(
    row["duplicate_copies_removed"] for row in observed_meta
)

assert total_source_rows == total_observed_rows + total_duplicates

summary = {
    "run_id": RUN_ID,
    "fingerprint": FINGERPRINT,
    "mode": "full_history",
    "source_files": len(sources),
    "original_files": sum(source["original"] for source in sources),
    "new_export_files": sum(not source["original"] for source in sources),
    "source_rows": total_source_rows,
    "original_source_rows": original_rows,
    "observed_rows": total_observed_rows,
    "duplicate_copies_removed": total_duplicates,
    "analysis_rows": sum(row["analysis_rows"] for row in analysis_results),
    "retained_real_rows": sum(
        row["retained_real_rows"] for row in analysis_results
    ),
    "synthetic_rows": sum(row["synthetic_rows"] for row in analysis_results),
    "first_timestamp_athens": fx.athens(FIRST_NS),
    "last_timestamp_athens": fx.athens(LAST_NS),
    "months_processed": len(observed_meta),
    "gap_limit_seconds_inclusive": MAX_GAP_SECONDS,
    "interpolation_step_seconds": INTERPOLATION_STEP_SECONDS,
    "interpolated_availability": "right real anchor B",
    "locked_observed_rows": sum(row["locked_rows"] for row in observed_meta),
    "both_sizes_zero_observed_rows": sum(
        row["both_sizes_zero_rows"] for row in observed_meta
    ),
    **dead_totals,
    "final_checks": check_totals,
    "full_history_machine_audit_passed": True,
    "size_units": "source units; absolute denomination unconfirmed",
    "boundary_anchor_policy": "provisional; anchors may be inside exclusions",
    "human_review": "pending",
    "source_stability_check": "file size and modification time; no content hashing",
    "coverage_basis": "between first and last observed quotes",
    "wall_minutes_since_settings_cell": round(
        (time.perf_counter() - STARTED) / 60, 3
    )
}

fx.dump_json(REPORT / "execution_summary.json", summary)
fx.dump_json(RUN / "run_state.json", {
    "status": "complete",
    "fingerprint": FINGERPRINT,
    "summary": str((REPORT / "execution_summary.json").relative_to(REPO)),
    "human_review": "pending"
})

FINAL_AUDIT_PASSED = True

print("NOTEBOOK_01_EXECUTION_SUMMARY")
print(json.dumps(summary, indent=2))

In [ ]:
# Update lightweight project records after the machine audit passes.
# This does not record human approval.
# The original Notebook 00 audit remains an audit of its original inputs.
# Its preliminary recovery tables are not the consolidated dataset.

assert FINAL_AUDIT_PASSED
assert fx.load_json(RUN / "run_state.json")["fingerprint"] == FINGERPRINT

docs = REPO / "docs"
docs.mkdir(exist_ok=True)

decision_text = f"""
## Notebook 01 — {RUN_ID}

- Decision: combine original and downloaded EURUSD exports.
  Rationale: use recovered observations before evaluating remaining gaps.
  Component: ETL.
  Status: agreed.

- Decision: collapse equal timestamps only when all four parsed quote fields agree.
  Rationale: remove overlapping copies while preserving later repeated quotes.
  Component: observed quotes.
  Status: agreed; conflicting values require review.

- Decision: gaps of 300 seconds or less qualify for linear interpolation.
  Rationale: agreed research treatment of short quote gaps.
  Component: analysis output.
  Status: agreed.

- Decision: synthetic timestamps start 10 seconds after A; B remains fixed.
  Rationale: preserve the agreed spacing and anchors.
  Component: interpolation.
  Status: agreed.

- Decision: synthetic availability is B, with no execution use.
  Rationale: B is future information until it arrives.
  Component: features and historical replay.
  Status: agreed correctness requirement.

- Decision: raw gaps above 300 seconds are never filled.
  Rationale: keep dead zones unavailable to market-based analysis.
  Component: continuity.
  Status: agreed.

- Decision: exclude first two and last one New York session hours,
  expected weekends, January 1 in Athens and verified quote closures.
  Rationale: agreed analysis scope.
  Component: analysis output.
  Status: agreed.

- Decision: retain only allowed synthetic timestamps, while permitting anchors
  just inside excluded periods.
  Rationale: complete eligible boundary gaps using real observations.
  Component: interpolation boundaries.
  Status: provisional assumption for human review.

- Decision: retain locked quotes and zero displayed sizes with flags;
  invalid inputs stop processing.
  Rationale: preserve evidence without silently inventing replacement quotes.
  Component: quote quality.
  Status: conservative implementation; human review pending.

- Decision: retain source size units.
  Rationale: absolute size denomination remains unconfirmed.
  Component: sizes.
  Status: provisional.

- Validation: full-history machine checks passed for this run.
  Human review: pending.
"""

decision_file = docs / "design_decisions.md"
existing = decision_file.read_text(encoding="utf-8") if decision_file.exists() else "# Design decisions\n"
marker = f"## Notebook 01 — {RUN_ID}"
if marker not in existing:
    decision_file.write_text(
        existing.rstrip() + "\n\n" + decision_text.strip() + "\n",
        encoding="utf-8"
    )

state_text = f"""# Project state

- Current stage: Notebook 01 execution complete; human review pending.
- Notebook 00: full original-history audit executed.
- Notebook 01: machine checks passed for run `{RUN_ID}`.
- Human-reviewed completion of Notebook 01: pending.
- Originals: `data/`.
- Downloaded exports: `data/2017 2018 investigation/`.
- Observed output: `data/processed/{RUN_ID}/observed/`.
- Analysis output: `data/processed/{RUN_ID}/analysis/`.
- Source provenance: `data/processed/{RUN_ID}/_staging/`.
- Reports: `reports/tables/etl/{RUN_ID}/`.
- Required run state: `data/processed/{RUN_ID}/run_state.json`.

## Outstanding review

- Review remaining dead zones and their dates.
- Review locked quotes and zero displayed sizes.
- Confirm absolute displayed-size denomination.
- Review the provisional treatment of anchors inside excluded hours.
- Exact historical holiday quote-closure coverage remains provisional.
- Interpolated observations must retain right-anchor availability.

## Dependencies and stale outputs

- Notebook 00 remains valid for its original seven-file snapshot.
- Preliminary recovery tables are superseded for consolidated-data use.
- Do not combine outputs from different ETL run identifiers.
- Regenerate any downstream output built from an older quote history.
- This notebook creates no candles, indicators, models or trading policies.

## Next step

Review Notebook 01 outputs with Raymond before advancing.
"""

(docs / "project_state.md").write_text(state_text, encoding="utf-8")

readme_path = REPO / "README.md"
readme = readme_path.read_text(encoding="utf-8") if readme_path.exists() else (
    "# Inventory-Aware FX Pricing and Hedging Simulator\n"
)

methodology = """## Current methodology

Notebook 00 audits the original full history. Notebook 01 combines original
and downloaded EURUSD quotes, preserving source provenance.

The observed dataset remains separate from the interpolated analysis dataset.
Eligible real-quote gaps are at most 300 seconds. Interior points are spaced
10 seconds from the left anchor and become available at the right anchor.
Longer gaps remain unfilled and break analysis continuity.

Research exclusions and continuity identifiers are stored explicitly.
Human review of Notebook 01 remains pending.

"""

start = readme.find("## Current methodology")
end = readme.find("\n## ", start + 1) if start >= 0 else -1

if start >= 0:
    readme = readme[:start] + methodology + (readme[end + 1:] if end >= 0 else "")
else:
    readme = readme.rstrip() + "\n\n" + methodology

disclosure = (
    "Development approach: This is an AI-assisted research project. "
    "Raymond Fendi defined the business objectives and provided FX domain "
    "direction. OpenAI Codex assisted with implementation, validation "
    "code, and documentation. CONTRIBUTIONS.md records completed design "
    "decisions, reviews, and validation."
)

required_text = [
    "Results pending execution and validation.",
    "This is a research simulator.",
    "Client behavior is synthetic unless actual client data is supplied.",
    "Displayed sizes are not executed trade volumes.",
    "One feed does not represent the entire FX market.",
    "Predictive accuracy does not establish trading profitability.",
    "Simulated profitability depends on the modeling assumptions.",
    disclosure
]

for text in required_text:
    if text not in readme:
        readme = readme.rstrip() + "\n\n" + text + "\n"

readme_path.write_text(readme, encoding="utf-8")

contributions_path = REPO / "CONTRIBUTIONS.md"
contributions = (
    contributions_path.read_text(encoding="utf-8")
    if contributions_path.exists() else "# Contributions\n"
)

contribution_entry = f"""
## Notebook 01 execution — {RUN_ID}

- Raymond Fendi defined the FX research objectives and processing direction.
- OpenAI Codex supplied notebook implementation, validation code and documentation.
- The executed notebook reports passing full-history machine checks.
- Raymond's review and acceptance of these outputs remain pending.
"""

if f"## Notebook 01 execution — {RUN_ID}" not in contributions:
    contributions_path.write_text(
        contributions.rstrip() + "\n\n" + contribution_entry.strip() + "\n",
        encoding="utf-8"
    )

# Ensure generated datasets and reports remain outside Git tracking.
ignore_path = REPO / ".gitignore"
ignore_text = ignore_path.read_text(encoding="utf-8") if ignore_path.exists() else ""
for rule in [
    "data/", "outputs/", "reports/tables/", "reports/figures/",
    ".ipynb_checkpoints/", "__pycache__/", "*.py[cod]", "*.duckdb", "*.duckdb.wal"
]:
    if rule not in ignore_text.splitlines():
        ignore_text = ignore_text.rstrip() + "\n" + rule + "\n"
ignore_path.write_text(ignore_text, encoding="utf-8")

print("Project records updated. Human review remains pending.")

In [ ]:
# Human-readable interpolation example.
# Show the real anchors separately from their synthetic interior points.

analysis_paths = [row["path"] for row in analysis_results]
con.read_parquet(
    analysis_paths, hive_partitioning=False
).create_view("all_analysis", replace=True)

example = fx.record(con, """
    SELECT left_anchor_ns, right_anchor_ns
    FROM all_analysis
    WHERE is_interpolated
    ORDER BY ts_ns
    LIMIT 1
""")

if example is not None:
    a = example["left_anchor_ns"]
    b = example["right_anchor_ns"]

    anchors = con.execute(f"""
        SELECT ts_ns, bid, ask, bid_size, ask_size
        FROM all_observed WHERE ts_ns IN ({a}, {b})
        ORDER BY ts_ns
    """).df()
    anchors["timestamp_athens"] = anchors["ts_ns"].map(fx.athens)

    interior = con.execute(f"""
        SELECT
            ts_ns, available_ns, bid, ask, bid_size, ask_size,
            is_interpolated, data_segment_id, scope_id
        FROM all_analysis
        WHERE is_interpolated
          AND left_anchor_ns = {a}
          AND right_anchor_ns = {b}
        ORDER BY ts_ns
    """).df()
    interior["timestamp_athens"] = interior["ts_ns"].map(fx.athens)
    interior["available_at_athens"] = interior["available_ns"].map(fx.athens)

    anchors.to_csv(REPORT / "example_real_anchors.csv", index=False)
    interior.to_csv(REPORT / "example_interpolated_points.csv", index=False)

    print("INTERPOLATION_REAL_ANCHORS")
    display(anchors.drop(columns="ts_ns"))

    print("INTERPOLATION_SYNTHETIC_POINTS")
    display(interior.drop(columns=["ts_ns", "available_ns"]))
else:
    print("No synthetic points were required inside the retained analysis scope.")

In [ ]:
# Return these outputs for review:
#
# 1. NOTEBOOK_01_EXECUTION_SUMMARY.
# 2. source_summary.csv, or its displayed dataframe.
# 3. monthly_reconciliation.csv, or its displayed dataframe.
# 4. The final audit totals from Cell 16.
# 5. dead_zone_monthly.csv.
# 6. The largest dead-zone table from Cell 17.
# 7. The real-anchor and synthetic-point example from Cell 21.
#
# If processing stops:
# - Return the traceback.
# - Return the named invalid-quote or conflicting-timestamp evidence.
# - Do not continue to later cells.
#
# Before committing:
# - Clear notebook outputs containing local paths or quote examples.
#
# Stop after this notebook.
# Review these outputs before building sampling, candles or indicators.

print("Reports:", REPORT)
print("Observed quotes:", RUN / "observed")
print("Analysis series:", RUN / "analysis")
print("Human review: pending")
print("STOP HERE: return the requested outputs before the next component.")

con.close()